# Visualizations & Evaluation Metrics

This notebook produces every figure used in the project report with **seaborn**, saves each
one as a PNG under `figures/`, and explains the evaluation metrics. Run it top to bottom;
the figures are regenerated and saved automatically.

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.05)
TEAL, RED, GREY = "#0f766e", "#ef4444", "#94a3b8"

os.makedirs("figures", exist_ok=True)

In [ ]:
# Load data and the saved model-comparison results
df = pd.read_csv("data/final_data.csv")
cmp = pd.read_csv("data/model_comparison_final_results.csv").sort_values("mae").reset_index(drop=True)

# Short, readable model names + a colour for each (BERTurk = teal, Hybrid = red, others = grey)
cmp["short"] = (cmp["model"]
    .str.replace(" Regression", "", regex=False)
    .str.replace(" (BERT features)", "", regex=False)
    .str.replace("Sentence-BERT", "SBERT", regex=False)
    .str.replace("CountVectorizer", "CountVec", regex=False)
    .str.replace("BERTurk Fine-Tuned", "BERTurk FT", regex=False))
pal = {s: ("#0f766e" if "BERTurk" in m else RED if "Hybrid" in m else GREY)
       for s, m in zip(cmp["short"], cmp["model"])}

def label_all(ax, fmt="%.2f"):
    for c in ax.containers:
        ax.bar_label(c, fmt=fmt, fontsize=8, padding=2)

cmp

## Evaluation Metrics — what each one means

All three metrics compare the **predicted** quality score with the **true** score on the
held-out test set. Let $y_i$ be the true score, $\hat{y}_i$ the prediction and $n$ the number
of conversations.

**MAE — Mean Absolute Error**

$$\mathrm{MAE} = \frac{1}{n}\sum_{i=1}^{n} \lvert y_i - \hat{y}_i \rvert$$

The average absolute difference between prediction and truth, in the same 0–100 unit as the
score. It answers *"on average, by how many points is the model wrong?"* **Lower is better.**

**RMSE — Root Mean Squared Error**

$$\mathrm{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$$

Like MAE but squares the errors first, so **large mistakes are punished much more** than small
ones. Also in score units, and always $\mathrm{RMSE} \ge \mathrm{MAE}$; a big gap between the
two means a few large errors. **Lower is better.**

**R² — Coefficient of Determination**

$$R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}$$

The fraction of the variance in the true scores that the model explains. $R^2 = 1$ is perfect,
$R^2 = 0$ is no better than always predicting the mean, and **negative $R^2$ is worse than the
mean**. It is unit-free, which makes it easy to compare across datasets. **Higher is better.**

## 1. Dataset class distribution

How many conversations fall into each quality class. This shows the data is **imbalanced**
toward *Good* calls (1295) versus *average* and *poor* (700 each) — important context, because
a model could look accurate just by favouring the majority class.

In [ ]:
plt.figure(figsize=(6, 3.8))
ax = sns.countplot(data=df, x="expected_quality_class", order=["Good", "average", "poor"],
                   hue="expected_quality_class",
                   palette={"Good": "#0d9488", "average": "#5ec9b5", "poor": "#f59e0b"}, legend=False)
label_all(ax, "%d")
ax.set(xlabel="Quality class", ylabel="Conversations", title="Dataset class distribution (n=2695)")
plt.tight_layout()
plt.savefig("figures/fig_class_distribution.png", dpi=150)
plt.show()

## 2. Distribution of ground-truth scores

The histogram (with a smoothed KDE curve) shows how the continuous 0–100 scores are spread.
The dashed line marks the mean (55.4). A wide spread means the regression task is meaningful —
the model must distinguish a full range of quality, not just one value.

In [ ]:
plt.figure(figsize=(6, 3.8))
ax = sns.histplot(df["final_quality_score"], bins=25, kde=True, color="#0d9488")
ax.axvline(df["final_quality_score"].mean(), color=RED, ls="--", lw=1.5,
           label=f"mean = {df['final_quality_score'].mean():.1f}")
ax.set(xlabel="Final quality score (0-100)", ylabel="Frequency",
       title="Distribution of ground-truth scores")
ax.legend()
plt.tight_layout()
plt.savefig("figures/fig_score_distribution.png", dpi=150)
plt.show()

## 3. MAE per model

Average absolute error of each model, sorted best-to-worst. BERTurk (teal) has the lowest MAE
(7.35 points), meaning its predictions are on average closest to the true score.

In [ ]:
plt.figure(figsize=(7, 4))
ax = sns.barplot(data=cmp, x="short", y="mae", hue="short", palette=pal, legend=False)
label_all(ax)
ax.set(xlabel="", ylabel="MAE", title="Model comparison — MAE (lower is better)")
plt.xticks(rotation=40, ha="right", fontsize=9)
plt.tight_layout()
plt.savefig("figures/fig_mae.png", dpi=150)
plt.show()

## 4. RMSE per model

Root mean squared error. Because RMSE penalises large errors more heavily, the gap between a
model's RMSE and its MAE hints at how often it makes big mistakes.

In [ ]:
plt.figure(figsize=(7, 4))
ax = sns.barplot(data=cmp, x="short", y="rmse", hue="short", palette=pal, legend=False)
label_all(ax)
ax.set(xlabel="", ylabel="RMSE", title="Model comparison — RMSE (lower is better)")
plt.xticks(rotation=40, ha="right", fontsize=9)
plt.tight_layout()
plt.savefig("figures/fig_rmse.png", dpi=150)
plt.show()

## 5. R² per model

Variance explained by each model. BERTurk reaches 0.876 (explains ~88% of the variance), while
the LLR monitor is negative — worse than predicting the mean — confirming it is only useful as a
complementary signal, not a standalone scorer.

In [ ]:
plt.figure(figsize=(7, 4))
ax = sns.barplot(data=cmp, x="short", y="r2", hue="short", palette=pal, legend=False)
label_all(ax)
ax.set(xlabel="", ylabel="R\u00b2", title="Model comparison — R\u00b2 (higher is better)")
plt.xticks(rotation=40, ha="right", fontsize=9)
plt.tight_layout()
plt.savefig("figures/fig_r2.png", dpi=150)
plt.show()

## 6. Combined comparison (MAE, RMSE, R² side by side)

All three metrics in one figure for the report. MAE and RMSE share the "lower is better" scale;
R² is shown separately because it is bounded and "higher is better".

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, metric, title in zip(axes, ["mae", "rmse", "r2"],
        ["MAE (lower better)", "RMSE (lower better)", "R\u00b2 (higher better)"]):
    sns.barplot(data=cmp, x="short", y=metric, hue="short", palette=pal, legend=False, ax=ax)
    for c in ax.containers:
        ax.bar_label(c, fmt="%.2f", fontsize=8, padding=2)
    ax.set(xlabel="", ylabel=metric.upper(), title=title)
    ax.tick_params(axis="x", rotation=40)
    for t in ax.get_xticklabels():
        t.set_ha("right"); t.set_fontsize(8.5)
fig.suptitle("Model performance comparison on the held-out test set", fontweight="bold", y=1.03)
plt.tight_layout()
plt.savefig("figures/fig_model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Rule engine — reward vs penalty by category

Not a model metric, but it visualises the rule-based scoring scheme: the maximum positive
**bonus** (teal) each compliance category can add and the maximum **penalty** (red) it can
subtract. The Anomaly and Silence categories carry no bonus and large penalties, reflecting that
they exist only to punish bad behaviour.

In [ ]:
rule = pd.DataFrame({
    "category": ["Opening", "Identity", "Empathy", "Resolution/Closing", "Anomaly", "Silence/dead air"] * 2,
    "type": ["Max bonus"] * 6 + ["Max penalty"] * 6,
    "points": [4, 2, 3, 5, 0, 0,  8, 5, 11, 9, 63, 25],
})
plt.figure(figsize=(8, 4.2))
ax = sns.barplot(data=rule, y="category", x="points", hue="type", palette=["#0d9488", RED])
ax.set(xlabel="Points", ylabel="", title="Rule engine: reward vs penalty by category")
plt.tight_layout()
plt.savefig("figures/fig_rule_weights.png", dpi=150)
plt.show()

All figures are now saved under `figures/`:
`fig_class_distribution.png`, `fig_score_distribution.png`, `fig_mae.png`, `fig_rmse.png`,
`fig_r2.png`, `fig_model_comparison.png`, `fig_rule_weights.png`.